In [5]:
# baseline_experiment.py
# Real-ESRGAN baseline qualitative experiment script

import subprocess
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt


project_root = Path('/root/Real-ESRGAN')
python_exec = '/root/miniconda/envs/realesrgan310/bin/python'
inference_script = project_root / 'inference_realesrgan.py'

# Input folders
input_root = project_root / 'paper_inputs'
face_dir = input_root / 'face'
arch_dir = input_root / 'architecture'
natural_dir = input_root / 'natural'

# Output folders
output_root = project_root / 'paper_results' / 'shared_prior'
face_out = output_root / 'face'
arch_out = output_root / 'architecture'
natural_out = output_root / 'natural'

# Figure folder
fig_dir = project_root / 'paper_results' / 'figures'


def ensure_dirs():
    for d in [face_dir, arch_dir, natural_dir, face_out, arch_out, natural_out, fig_dir]:
        d.mkdir(parents=True, exist_ok=True)


def list_images(folder):
    exts = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.webp']
    files = []
    for ext in exts:
        files.extend(folder.glob(ext))
    return sorted(files)


def print_dataset_status():
    print('Project root:', project_root)
    print('Input root:', input_root)
    print('Output root:', output_root)
    print('-' * 60)
    for name, folder in [('face', face_dir), ('architecture', arch_dir), ('natural', natural_dir)]:
        imgs = list_images(folder)
        print(f'{name}: {len(imgs)} image(s) found')
        for p in imgs[:10]:
            print('   ', p.name)
    print('-' * 60)


def run_realesrgan(input_dir, output_dir):
    cmd = [
        python_exec,
        str(inference_script),
        '-n', 'RealESRGAN_x4plus',
        '-i', str(input_dir),
        '-o', str(output_dir),
        '--fp32'
    ]
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=project_root, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'Inference failed for {input_dir}')


def find_output_image(output_dir, input_name_stem):
    candidates = list(output_dir.glob(f'{input_name_stem}*'))
    candidates = [p for p in candidates if p.suffix.lower() in ['.png', '.jpg', '.jpeg', '.bmp', '.webp']]
    return sorted(candidates)[0] if candidates else None


def save_domain_figure(domain_name, input_dir, output_dir, save_path, max_items=3):
    inputs = list_images(input_dir)[:max_items]
    if len(inputs) == 0:
        print(f'No images found in {input_dir}, skip figure generation.')
        return

    fig, axes = plt.subplots(len(inputs), 2, figsize=(10, 4 * len(inputs)))
    if len(inputs) == 1:
        axes = [axes]

    for row, inp in enumerate(inputs):
        out = find_output_image(output_dir, inp.stem)

        inp_img = Image.open(inp).convert('RGB')
        axes[row][0].imshow(inp_img)
        axes[row][0].set_title('Input')
        axes[row][0].axis('off')

        if out is not None:
            out_img = Image.open(out).convert('RGB')
            axes[row][1].imshow(out_img)
            axes[row][1].set_title('Real-ESRGAN Output')
        else:
            axes[row][1].text(0.5, 0.5, 'Output not found', ha='center', va='center')
        axes[row][1].axis('off')

    plt.suptitle(f'{domain_name} Results under Shared Prior', fontsize=16)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print('Saved figure:', save_path)


def main():
    ensure_dirs()

    print('\nStep 1: Please put your test images into these folders if not already done:')
    print(face_dir)
    print(arch_dir)
    print(natural_dir)
    print()

    print_dataset_status()

    # Run baseline inference
    for input_dir, output_dir in [
        (face_dir, face_out),
        (arch_dir, arch_out),
        (natural_dir, natural_out),
    ]:
        if len(list_images(input_dir)) > 0:
            run_realesrgan(input_dir, output_dir)
        else:
            print(f'Skip {input_dir}, no images found.')

    print('\nStep 2: Generating thesis-ready figures...')
    save_domain_figure('Face', face_dir, face_out, fig_dir / 'face_shared_prior.png', max_items=3)
    save_domain_figure('Architecture', arch_dir, arch_out, fig_dir / 'architecture_shared_prior.png', max_items=3)
    save_domain_figure('Natural Scene', natural_dir, natural_out, fig_dir / 'natural_shared_prior.png', max_items=3)

    print('\nDone.')
    print('Generated figures are saved in:')
    print(fig_dir)
    print('\nSuggested LaTeX example:')
    print(r'''
\begin{figure}[t]
    \centering
    \includegraphics[width=0.9\linewidth]{figures/face_shared_prior.png}
    \caption{Qualitative results of the shared-prior Real-ESRGAN baseline on face images.}
    \label{fig:qual_face}
\end{figure}
''')


if __name__ == '__main__':
    main()


Step 1: Please put your test images into these folders if not already done:
/root/Real-ESRGAN/paper_inputs/face
/root/Real-ESRGAN/paper_inputs/architecture
/root/Real-ESRGAN/paper_inputs/natural

Project root: /root/Real-ESRGAN
Input root: /root/Real-ESRGAN/paper_inputs
Output root: /root/Real-ESRGAN/paper_results/shared_prior
------------------------------------------------------------
face: 5 image(s) found
    00017_gray.png
    0014.jpg
    0030.jpg
    children-alpha.png
    wolf_gray.jpg
architecture: 1 image(s) found
    OST_009.png
natural: 3 image(s) found
    00003.png
    ADE_val_00000114.jpg
    tree_alpha_16bit.png
------------------------------------------------------------
Running: /root/miniconda/envs/realesrgan310/bin/python /root/Real-ESRGAN/inference_realesrgan.py -n RealESRGAN_x4plus -i /root/Real-ESRGAN/paper_inputs/face -o /root/Real-ESRGAN/paper_results/shared_prior/face --fp32
Testing 0 00017_gray
Testing 1 0014
Testing 2 0030
Testing 3 children-alpha
Testing 4

In [6]:
# category_comparison_experiment.py
# Second experiment for thesis:
# Compare Real-ESRGAN outputs under three domain folders and create a single paper-ready summary figure.
#
# What this script does:
# 1. Reads images from paper_inputs/face, paper_inputs/architecture, paper_inputs/natural
# 2. Runs the same Real-ESRGAN baseline on all images
# 3. Builds one compact comparison figure with 3 rows:
#    each row = Input | Restored | Zoomed crop
# 4. Saves a paper-ready figure:
#    /root/Real-ESRGAN/paper_results/figures/domain_comparison_panel.png
#
# This gives you another figure for the paper:
# - one clean multi-domain comparison panel
# - useful for "Qualitative and Qualitative Results" or "Ablation and Analysis"

import subprocess
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt


project_root = Path('/root/Real-ESRGAN')
python_exec = '/root/miniconda/envs/realesrgan310/bin/python'
inference_script = project_root / 'inference_realesrgan.py'

input_root = project_root / 'paper_inputs'
output_root = project_root / 'paper_results' / 'shared_prior'
fig_dir = project_root / 'paper_results' / 'figures'

domains = {
    'Face': {
        'input': input_root / 'face',
        'output': output_root / 'face'
    },
    'Architecture': {
        'input': input_root / 'architecture',
        'output': output_root / 'architecture'
    },
    'Natural Scene': {
        'input': input_root / 'natural',
        'output': output_root / 'natural'
    }
}


def ensure_dirs():
    fig_dir.mkdir(parents=True, exist_ok=True)
    for d in domains.values():
        d['input'].mkdir(parents=True, exist_ok=True)
        d['output'].mkdir(parents=True, exist_ok=True)


def list_images(folder):
    exts = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.webp']
    files = []
    for ext in exts:
        files.extend(folder.glob(ext))
    return sorted(files)


def run_realesrgan(input_dir, output_dir):
    imgs = list_images(input_dir)
    if len(imgs) == 0:
        print(f'Skip {input_dir}, no images found.')
        return

    cmd = [
        python_exec,
        str(inference_script),
        '-n', 'RealESRGAN_x4plus',
        '-i', str(input_dir),
        '-o', str(output_dir),
        '--fp32'
    ]
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=project_root, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'Inference failed for {input_dir}')


def find_output_image(output_dir, input_stem):
    candidates = list(output_dir.glob(f'{input_stem}*'))
    candidates = [p for p in candidates if p.suffix.lower() in ['.png', '.jpg', '.jpeg', '.bmp', '.webp']]
    return sorted(candidates)[0] if candidates else None


def add_crop_box(img, crop_box, color='red', width=4):
    img = img.copy()
    draw = ImageDraw.Draw(img)
    x1, y1, x2, y2 = crop_box
    for i in range(width):
        draw.rectangle([x1 - i, y1 - i, x2 + i, y2 + i], outline=color)
    return img


def center_crop_box(img, crop_ratio=0.35):
    w, h = img.size
    cw = int(w * crop_ratio)
    ch = int(h * crop_ratio)
    x1 = max((w - cw) // 2, 0)
    y1 = max((h - ch) // 2, 0)
    x2 = min(x1 + cw, w)
    y2 = min(y1 + ch, h)
    return (x1, y1, x2, y2)


def crop_and_resize(img, crop_box, out_size=(400, 400)):
    crop = img.crop(crop_box)
    return crop.resize(out_size, Image.Resampling.LANCZOS)


def pick_first_image_pair(input_dir, output_dir):
    inputs = list_images(input_dir)
    if len(inputs) == 0:
        return None, None
    inp = inputs[0]
    out = find_output_image(output_dir, inp.stem)
    return inp, out


def build_summary_panel(save_path):
    rows = []
    for domain_name, paths in domains.items():
        inp_path, out_path = pick_first_image_pair(paths['input'], paths['output'])
        if inp_path is None or out_path is None:
            print(f'Warning: missing pair for {domain_name}, skip this row.')
            continue
        rows.append((domain_name, inp_path, out_path))

    if len(rows) == 0:
        print('No valid image pairs found. Cannot create summary figure.')
        return

    fig, axes = plt.subplots(len(rows), 3, figsize=(12, 4 * len(rows)))
    if len(rows) == 1:
        axes = [axes]

    for r, (domain_name, inp_path, out_path) in enumerate(rows):
        inp_img = Image.open(inp_path).convert('RGB')
        out_img = Image.open(out_path).convert('RGB')

        crop_box_in = center_crop_box(inp_img, crop_ratio=0.35)
        crop_box_out = center_crop_box(out_img, crop_ratio=0.35)

        inp_boxed = add_crop_box(inp_img, crop_box_in, color='yellow', width=4)
        out_boxed = add_crop_box(out_img, crop_box_out, color='yellow', width=4)

        zoom_img = crop_and_resize(out_img, crop_box_out, out_size=(500, 500))

        axes[r][0].imshow(inp_boxed)
        axes[r][0].set_title(f'{domain_name} Input')
        axes[r][0].axis('off')

        axes[r][1].imshow(out_boxed)
        axes[r][1].set_title('Restored Output')
        axes[r][1].axis('off')

        axes[r][2].imshow(zoom_img)
        axes[r][2].set_title('Zoomed Detail')
        axes[r][2].axis('off')

    plt.suptitle('Multi-domain Qualitative Comparison under Shared Prior', fontsize=16)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print('Saved figure:', save_path)


def main():
    ensure_dirs()

    print('Checking input folders...')
    for domain_name, paths in domains.items():
        imgs = list_images(paths['input'])
        print(f'{domain_name}: {len(imgs)} image(s) found in {paths["input"]}')

    print('\nRunning inference if needed...')
    for domain_name, paths in domains.items():
        run_realesrgan(paths['input'], paths['output'])

    print('\nGenerating paper-ready summary figure...')
    save_path = fig_dir / 'domain_comparison_panel.png'
    build_summary_panel(save_path)

    print('\nDone.')
    print('You can insert this figure into LaTeX with:')
    print(r'''
\begin{figure*}[t]
    \centering
    \includegraphics[width=\textwidth]{figures/domain_comparison_panel.png}
    \caption{Qualitative comparison of Real-ESRGAN restoration results across face, architectural, and natural-scene images under the shared degradation prior. For each domain, we show the input image, the restored output, and a zoomed local detail for visual inspection of edge continuity, texture fidelity, and artifact behavior.}
    \label{fig:domain_comparison_panel}
\end{figure*}
''')


if __name__ == '__main__':
    main()

Checking input folders...
Face: 5 image(s) found in /root/Real-ESRGAN/paper_inputs/face
Architecture: 1 image(s) found in /root/Real-ESRGAN/paper_inputs/architecture
Natural Scene: 3 image(s) found in /root/Real-ESRGAN/paper_inputs/natural

Running inference if needed...
Running: /root/miniconda/envs/realesrgan310/bin/python /root/Real-ESRGAN/inference_realesrgan.py -n RealESRGAN_x4plus -i /root/Real-ESRGAN/paper_inputs/face -o /root/Real-ESRGAN/paper_results/shared_prior/face --fp32
Testing 0 00017_gray
Testing 1 0014
Testing 2 0030
Testing 3 children-alpha
Testing 4 wolf_gray

Running: /root/miniconda/envs/realesrgan310/bin/python /root/Real-ESRGAN/inference_realesrgan.py -n RealESRGAN_x4plus -i /root/Real-ESRGAN/paper_inputs/architecture -o /root/Real-ESRGAN/paper_results/shared_prior/architecture --fp32
Testing 0 OST_009

Running: /root/miniconda/envs/realesrgan310/bin/python /root/Real-ESRGAN/inference_realesrgan.py -n RealESRGAN_x4plus -i /root/Real-ESRGAN/paper_inputs/natural -o

In [7]:
# degradation_sensitivity_experiment.py
# Sensitivity experiment:
# For each domain (face / architecture / natural), choose 1 image,
# synthesize three degradation levels (mild / moderate / severe),
# run Real-ESRGAN on each degraded image,
# and create a thesis-ready panel figure.
#
# Output figure:
# /root/Real-ESRGAN/paper_results/figures/degradation_sensitivity_panel.png
#
# Input:
# Put at least one image into each folder:
# /root/Real-ESRGAN/paper_inputs/face
# /root/Real-ESRGAN/paper_inputs/architecture
# /root/Real-ESRGAN/paper_inputs/natural

import io
import random
import subprocess
from pathlib import Path

import numpy as np
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt


# -----------------------------
# Basic paths
# -----------------------------
project_root = Path('/root/Real-ESRGAN')
python_exec = '/root/miniconda/envs/realesrgan310/bin/python'
inference_script = project_root / 'inference_realesrgan.py'

input_root = project_root / 'paper_inputs'
fig_dir = project_root / 'paper_results' / 'figures'
exp_root = project_root / 'paper_results' / 'degradation_sensitivity'

domains = {
    'Face': {
        'input': input_root / 'face',
        'work': exp_root / 'face'
    },
    'Architecture': {
        'input': input_root / 'architecture',
        'work': exp_root / 'architecture'
    },
    'Natural Scene': {
        'input': input_root / 'natural',
        'work': exp_root / 'natural'
    }
}

levels = ['mild', 'moderate', 'severe']


# -----------------------------
# Helpers
# -----------------------------
def ensure_dirs():
    fig_dir.mkdir(parents=True, exist_ok=True)
    exp_root.mkdir(parents=True, exist_ok=True)
    for item in domains.values():
        item['input'].mkdir(parents=True, exist_ok=True)
        item['work'].mkdir(parents=True, exist_ok=True)
        (item['work'] / 'degraded').mkdir(parents=True, exist_ok=True)
        (item['work'] / 'restored').mkdir(parents=True, exist_ok=True)


def list_images(folder):
    exts = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.webp']
    files = []
    for ext in exts:
        files.extend(folder.glob(ext))
    return sorted(files)


def pick_first_image(folder):
    imgs = list_images(folder)
    return imgs[0] if len(imgs) > 0 else None


def resize_if_too_large(img, max_side=1200):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    if w >= h:
        new_w = max_side
        new_h = int(h * max_side / w)
    else:
        new_h = max_side
        new_w = int(w * max_side / h)
    return img.resize((new_w, new_h), Image.Resampling.LANCZOS)


# -----------------------------
# Degradation functions
# -----------------------------
def add_gaussian_noise(img, sigma):
    arr = np.asarray(img).astype(np.float32)
    noise = np.random.normal(0, sigma, arr.shape).astype(np.float32)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)


def jpeg_compress(img, quality):
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    return Image.open(buf).convert('RGB')


def downsample_upsample(img, scale_factor):
    w, h = img.size
    new_w = max(8, int(w / scale_factor))
    new_h = max(8, int(h / scale_factor))
    small = img.resize((new_w, new_h), Image.Resampling.BICUBIC)
    back = small.resize((w, h), Image.Resampling.BICUBIC)
    return back


def degrade_image(img, level):
    img = img.convert('RGB')
    img = resize_if_too_large(img, max_side=1200)

    if level == 'mild':
        # light blur + light noise + mild jpeg + small resampling loss
        out = img.filter(ImageFilter.GaussianBlur(radius=0.8))
        out = add_gaussian_noise(out, sigma=4)
        out = downsample_upsample(out, scale_factor=1.5)
        out = jpeg_compress(out, quality=70)
        return out

    elif level == 'moderate':
        # stronger blur + more noise + heavier jpeg + larger resampling loss
        out = img.filter(ImageFilter.GaussianBlur(radius=1.5))
        out = add_gaussian_noise(out, sigma=8)
        out = downsample_upsample(out, scale_factor=2.0)
        out = jpeg_compress(out, quality=45)
        return out

    elif level == 'severe':
        # strong blur + heavy noise + strong jpeg + larger resampling loss
        out = img.filter(ImageFilter.GaussianBlur(radius=2.2))
        out = add_gaussian_noise(out, sigma=14)
        out = downsample_upsample(out, scale_factor=3.0)
        out = jpeg_compress(out, quality=25)
        return out

    else:
        raise ValueError(f'Unknown degradation level: {level}')


# -----------------------------
# Real-ESRGAN inference
# -----------------------------
def run_realesrgan(input_path, output_dir):
    cmd = [
        python_exec,
        str(inference_script),
        '-n', 'RealESRGAN_x4plus',
        '-i', str(input_path),
        '-o', str(output_dir),
        '--fp32'
    ]
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=project_root, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'Inference failed for {input_path}')


def find_output_image(output_dir, input_stem):
    candidates = list(output_dir.glob(f'{input_stem}*'))
    candidates = [p for p in candidates if p.suffix.lower() in ['.png', '.jpg', '.jpeg', '.bmp', '.webp']]
    return sorted(candidates)[0] if candidates else None


# -----------------------------
# Build degraded inputs and restored outputs
# -----------------------------
def prepare_domain_images():
    summary = {}

    for domain_name, item in domains.items():
        src = pick_first_image(item['input'])
        if src is None:
            print(f'No input image found for {domain_name} in {item["input"]}')
            summary[domain_name] = None
            continue

        print(f'Using source image for {domain_name}: {src.name}')
        orig_img = Image.open(src).convert('RGB')
        orig_img = resize_if_too_large(orig_img, max_side=1200)

        degraded_dir = item['work'] / 'degraded'
        restored_dir = item['work'] / 'restored'

        domain_info = {
            'original': src,
            'degraded': {},
            'restored': {}
        }

        for level in levels:
            degraded_img = degrade_image(orig_img, level)
            degraded_path = degraded_dir / f'{src.stem}_{level}.png'
            degraded_img.save(degraded_path)
            print('Saved degraded image:', degraded_path)

            run_realesrgan(degraded_path, restored_dir)

            restored_path = find_output_image(restored_dir, degraded_path.stem)
            if restored_path is None:
                raise RuntimeError(f'Cannot find restored output for {degraded_path}')
            print('Found restored image:', restored_path)

            domain_info['degraded'][level] = degraded_path
            domain_info['restored'][level] = restored_path

        summary[domain_name] = domain_info

    return summary


# -----------------------------
# Figure creation
# -----------------------------
def make_panel(summary, save_path):
    available_domains = [k for k, v in summary.items() if v is not None]
    if len(available_domains) == 0:
        print('No valid domains found. Cannot build figure.')
        return

    # rows = domain count
    # cols = 1 original + 3 degraded + 3 restored = 7
    fig, axes = plt.subplots(len(available_domains), 7, figsize=(21, 4.5 * len(available_domains)))

    if len(available_domains) == 1:
        axes = [axes]

    col_titles = [
        'Original',
        'Mild Degradation',
        'Moderate Degradation',
        'Severe Degradation',
        'Restored (Mild)',
        'Restored (Moderate)',
        'Restored (Severe)'
    ]

    for r, domain_name in enumerate(available_domains):
        info = summary[domain_name]

        orig_img = Image.open(info['original']).convert('RGB')
        mild_deg = Image.open(info['degraded']['mild']).convert('RGB')
        mod_deg = Image.open(info['degraded']['moderate']).convert('RGB')
        sev_deg = Image.open(info['degraded']['severe']).convert('RGB')
        mild_res = Image.open(info['restored']['mild']).convert('RGB')
        mod_res = Image.open(info['restored']['moderate']).convert('RGB')
        sev_res = Image.open(info['restored']['severe']).convert('RGB')

        imgs = [orig_img, mild_deg, mod_deg, sev_deg, mild_res, mod_res, sev_res]

        for c, im in enumerate(imgs):
            axes[r][c].imshow(im)
            if r == 0:
                axes[r][c].set_title(col_titles[c], fontsize=11)
            axes[r][c].axis('off')

        axes[r][0].set_ylabel(domain_name, fontsize=12)

    plt.suptitle('Sensitivity to Degradation Severity under Shared Prior', fontsize=16)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print('Saved figure:', save_path)


# -----------------------------
# Main
# -----------------------------
def main():
    random.seed(0)
    np.random.seed(0)

    ensure_dirs()

    print('Checking input folders...')
    for domain_name, item in domains.items():
        imgs = list_images(item['input'])
        print(f'{domain_name}: {len(imgs)} image(s) found in {item["input"]}')
        if len(imgs) > 0:
            print(f'  first image: {imgs[0].name}')

    print('\nPreparing degraded inputs and restored outputs...')
    summary = prepare_domain_images()

    print('\nGenerating thesis-ready figure...')
    save_path = fig_dir / 'degradation_sensitivity_panel.png'
    make_panel(summary, save_path)

    print('\nDone.')
    print('Figure saved to:')
    print(save_path)

    print('\nSuggested LaTeX:')
    print(r'''
\begin{figure*}[t]
    \centering
    \includegraphics[width=\textwidth]{figures/degradation_sensitivity_panel.png}
    \caption{Sensitivity analysis of the shared-prior Real-ESRGAN backbone under increasing degradation severity. For each image domain, we show the original image, three degraded inputs (mild, moderate, and severe), and the corresponding restoration results. As degradation becomes stronger, the fixed shared prior leads to more visible oversmoothing, texture instability, and structural distortion, especially under moderate-to-severe corruption.}
    \label{fig:degradation_sensitivity}
\end{figure*}
''')

    print('\nNotes:')
    print('- Put at least one image into each domain folder before running.')
    print('- This script automatically uses the first image found in each folder.')


if __name__ == '__main__':
    main()

Checking input folders...
Face: 5 image(s) found in /root/Real-ESRGAN/paper_inputs/face
  first image: 00017_gray.png
Architecture: 1 image(s) found in /root/Real-ESRGAN/paper_inputs/architecture
  first image: OST_009.png
Natural Scene: 3 image(s) found in /root/Real-ESRGAN/paper_inputs/natural
  first image: 00003.png

Preparing degraded inputs and restored outputs...
Using source image for Face: 00017_gray.png
Saved degraded image: /root/Real-ESRGAN/paper_results/degradation_sensitivity/face/degraded/00017_gray_mild.png
Running: /root/miniconda/envs/realesrgan310/bin/python /root/Real-ESRGAN/inference_realesrgan.py -n RealESRGAN_x4plus -i /root/Real-ESRGAN/paper_results/degradation_sensitivity/face/degraded/00017_gray_mild.png -o /root/Real-ESRGAN/paper_results/degradation_sensitivity/face/restored --fp32
Testing 0 00017_gray_mild

Found restored image: /root/Real-ESRGAN/paper_results/degradation_sensitivity/face/restored/00017_gray_mild_out.png
Saved degraded image: /root/Real-ESRG